# 01 Data and ML Pipeline Clean Notebook v7

This notebook is rebuilt from the successful experiment notebook. It starts from raw `ibtracs_NA.csv` when the curated event-node file is missing, preserves the final without-SEASON ML/risk-matrix logic, and includes Windows-safe single-threaded sklearn/joblib settings.


In [ ]:
# Cell 00: Global setup and imports
# Cell 00: Global setup and imports

import os
from pathlib import Path

# Windows + Chinese path safety for joblib/sklearn parallel backend.
import tempfile
JOBLIB_TEMP = Path(tempfile.gettempdir()) / "hurricane_resilience_joblib"
JOBLIB_TEMP.mkdir(parents=True, exist_ok=True)

os.environ["JOBLIB_TEMP_FOLDER"] = str(JOBLIB_TEMP)
os.environ["TMP"] = str(JOBLIB_TEMP)
os.environ["TEMP"] = str(JOBLIB_TEMP)

SKLEARN_N_JOBS = 1
RANDOM_STATE = 42

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold
)

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except ImportError:
    HAS_STRATIFIED_GROUP_KFOLD = False

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix
)

from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


# Optional model package availability flags.
# These flags are used by the seven-model benchmark section.
try:
    import xgboost  # noqa: F401
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    import lightgbm  # noqa: F401
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs_01_ml_pipeline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports completed.")
print("Project folder:", BASE_DIR)
print("StratifiedGroupKFold available:", HAS_STRATIFIED_GROUP_KFOLD)
print("SKLEARN_N_JOBS:", SKLEARN_N_JOBS)
print("XGBoost available:", HAS_XGBOOST)
print("LightGBM available:", HAS_LIGHTGBM)


In [ ]:
# Cell 01: Rebuild event-node dataset from raw IBTrACS if missing
# Cell 01: Rebuild event-node dataset from raw IBTrACS if missing

from pathlib import Path

RAW_IBTRACS_FILE = "ibtracs_NA.csv"
EVENT_NODE_FILE = "storm_node_dataset_14nodes_200km_50kt.csv"

raw_path = BASE_DIR / RAW_IBTRACS_FILE
event_node_path = BASE_DIR / EVENT_NODE_FILE

nodes = pd.DataFrame([
    {"node_id": "N1",  "node_name": "Houston",         "node_type": "Port",      "node_latitude": 29.7604, "node_longitude": -95.3698, "coastal": 1},
    {"node_id": "N2",  "node_name": "New Orleans",     "node_type": "Port",      "node_latitude": 29.9511, "node_longitude": -90.0715, "coastal": 1},
    {"node_id": "N3",  "node_name": "Miami",           "node_type": "Market",    "node_latitude": 25.7617, "node_longitude": -80.1918, "coastal": 1},
    {"node_id": "N4",  "node_name": "Orlando",         "node_type": "Warehouse", "node_latitude": 28.5383, "node_longitude": -81.3792, "coastal": 0},
    {"node_id": "N5",  "node_name": "Atlanta",         "node_type": "Warehouse", "node_latitude": 33.7490, "node_longitude": -84.3880, "coastal": 0},
    {"node_id": "N6",  "node_name": "Dallas",          "node_type": "Supplier",  "node_latitude": 32.7767, "node_longitude": -96.7970, "coastal": 0},
    {"node_id": "N7",  "node_name": "Tampa",           "node_type": "Market",    "node_latitude": 27.9506, "node_longitude": -82.4572, "coastal": 1},
    {"node_id": "N8",  "node_name": "Jacksonville",    "node_type": "Port",      "node_latitude": 30.3322, "node_longitude": -81.6557, "coastal": 1},
    {"node_id": "N9",  "node_name": "Mobile",          "node_type": "Port",      "node_latitude": 30.6954, "node_longitude": -88.0399, "coastal": 1},
    {"node_id": "N10", "node_name": "Savannah",        "node_type": "Port",      "node_latitude": 32.0809, "node_longitude": -81.0912, "coastal": 1},
    {"node_id": "N11", "node_name": "Charleston",      "node_type": "Port",      "node_latitude": 32.7765, "node_longitude": -79.9311, "coastal": 1},
    {"node_id": "N12", "node_name": "Corpus Christi",  "node_type": "Port",      "node_latitude": 27.8006, "node_longitude": -97.3964, "coastal": 1},
    {"node_id": "N13", "node_name": "Baton Rouge",     "node_type": "Warehouse", "node_latitude": 30.4515, "node_longitude": -91.1871, "coastal": 0},
    {"node_id": "N14", "node_name": "Charlotte",       "node_type": "Warehouse", "node_latitude": 35.2271, "node_longitude": -80.8431, "coastal": 0},
])

nodes.to_csv("supply_chain_nodes_14.csv", index=False, encoding="utf-8-sig")

def haversine_km(lat1, lon1, lat2, lon2):
    radius_earth_km = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    return 2 * radius_earth_km * np.arcsin(np.sqrt(a))

def rebuild_event_node_from_raw(raw_file):
    usecols = [
        "SID", "SEASON", "BASIN", "NAME", "ISO_TIME",
        "LAT", "LON", "WMO_WIND", "USA_WIND", "STORM_SPEED"
    ]

    raw = pd.read_csv(
        raw_file,
        usecols=lambda c: c in usecols,
        skiprows=[1],
        keep_default_na=False,
        low_memory=False
    )

    for col in ["SEASON", "LAT", "LON", "WMO_WIND", "USA_WIND", "STORM_SPEED"]:
        raw[col] = pd.to_numeric(raw[col].replace("", np.nan), errors="coerce")

    raw["ISO_TIME"] = pd.to_datetime(raw["ISO_TIME"], errors="coerce")
    raw["WIND_KTS"] = raw["USA_WIND"].where(raw["USA_WIND"].notna(), raw["WMO_WIND"])

    clean_tracks = raw[
        (raw["BASIN"] == "NA")
        & (raw["SEASON"] >= 2000)
        & (raw["SID"].astype(str).str.len() > 0)
        & raw["LAT"].notna()
        & raw["LON"].notna()
        & raw["WIND_KTS"].notna()
        & raw["ISO_TIME"].notna()
    ].copy()

    clean_tracks = clean_tracks.sort_values(["SID", "ISO_TIME"]).reset_index(drop=True)
    clean_tracks.to_csv("ibtracs_NA_clean_2000.csv", index=False, encoding="utf-8-sig")

    rows = []
    threshold_work_rows = []

    for sid, g in clean_tracks.groupby("SID", sort=False):
        g = g.sort_values("ISO_TIME").copy()

        season = int(g["SEASON"].iloc[0])
        name = str(g["NAME"].iloc[0])

        duration_hours = (
            g["ISO_TIME"].max() - g["ISO_TIME"].min()
        ).total_seconds() / 3600.0

        max_wind_kts = float(g["WIND_KTS"].max())
        mean_wind_kts = float(g["WIND_KTS"].mean())

        mean_storm_speed = float(g["STORM_SPEED"].mean()) if g["STORM_SPEED"].notna().any() else np.nan
        max_storm_speed = float(g["STORM_SPEED"].max()) if g["STORM_SPEED"].notna().any() else np.nan

        max_wind_idx = g["WIND_KTS"].idxmax()
        month = int(g.loc[max_wind_idx, "ISO_TIME"].month)

        storm_lats = g["LAT"].to_numpy(dtype=float)
        storm_lons = g["LON"].to_numpy(dtype=float)
        storm_winds = g["WIND_KTS"].to_numpy(dtype=float)

        for _, node in nodes.iterrows():
            distances = haversine_km(
                storm_lats,
                storm_lons,
                float(node["node_latitude"]),
                float(node["node_longitude"])
            )

            closest_idx = int(np.nanargmin(distances))

            def max_wind_within(radius_km):
                mask = distances <= radius_km
                if not np.any(mask):
                    return 0.0
                return float(np.nanmax(storm_winds[mask]))

            max_wind_within_50km = max_wind_within(50)
            max_wind_within_100km = max_wind_within(100)
            max_wind_within_150km = max_wind_within(150)
            max_wind_within_200km = max_wind_within(200)

            row = {
                "SID": sid,
                "SEASON": season,
                "NAME": name,
                "node_id": node["node_id"],
                "node_name": node["node_name"],
                "node_type": node["node_type"],
                "node_latitude": float(node["node_latitude"]),
                "node_longitude": float(node["node_longitude"]),
                "coastal": int(node["coastal"]),
                "month": month,
                "duration_hours": duration_hours,
                "max_wind_kts": max_wind_kts,
                "mean_wind_kts": mean_wind_kts,
                "mean_storm_speed": mean_storm_speed,
                "max_storm_speed": max_storm_speed,
                "min_distance_km": float(distances[closest_idx]),
                "wind_at_closest": float(storm_winds[closest_idx]),
                "max_wind_within_100km": max_wind_within_100km,
                "max_wind_within_200km": max_wind_within_200km,
                "distance_threshold_km": 200,
                "wind_threshold_kts": 50,
                "disrupted": int(max_wind_within_200km >= 50),
            }

            rows.append(row)

            threshold_work_row = row.copy()
            threshold_work_row["max_wind_within_50km"] = max_wind_within_50km
            threshold_work_row["max_wind_within_150km"] = max_wind_within_150km
            threshold_work_rows.append(threshold_work_row)

    event_node_df = pd.DataFrame(rows)
    threshold_work_df = pd.DataFrame(threshold_work_rows)

    event_node_df.to_csv(EVENT_NODE_FILE, index=False, encoding="utf-8-sig")

    threshold_rows = []
    distance_columns = {
        50: "max_wind_within_50km",
        100: "max_wind_within_100km",
        150: "max_wind_within_150km",
        200: "max_wind_within_200km",
    }

    for distance_km, wind_col in distance_columns.items():
        for wind_threshold_kts in [34, 50, 64]:
            label = (threshold_work_df[wind_col] >= wind_threshold_kts).astype(int)
            threshold_rows.append({
                "distance_threshold_km": distance_km,
                "wind_threshold_kts": wind_threshold_kts,
                "positive_count": int(label.sum()),
                "total_count": int(len(label)),
                "positive_ratio": float(label.mean()),
            })

    threshold_summary = pd.DataFrame(threshold_rows).sort_values(
        ["wind_threshold_kts", "distance_threshold_km"]
    ).reset_index(drop=True)

    threshold_summary.to_csv("threshold_summary_14nodes.csv", index=False, encoding="utf-8-sig")

    print("Clean track records:", clean_tracks.shape[0])
    print("Hurricane events:", clean_tracks["SID"].nunique())
    print("Saved:", EVENT_NODE_FILE)
    print("Event-node shape:", event_node_df.shape)
    print("Positive exposure count:", int(event_node_df["disrupted"].sum()))

    return event_node_df

if not event_node_path.exists():
    if not raw_path.exists():
        raise FileNotFoundError(
            f"Cannot find {EVENT_NODE_FILE} or {RAW_IBTRACS_FILE} in {BASE_DIR}."
        )
    print(f"{EVENT_NODE_FILE} not found. Rebuilding from {RAW_IBTRACS_FILE}.")
    df_rebuilt = rebuild_event_node_from_raw(raw_path)
else:
    print(f"Found existing {EVENT_NODE_FILE}; raw rebuild skipped.")

# Validate that the curated dataset exists now.
if not event_node_path.exists():
    raise FileNotFoundError(f"Failed to create {EVENT_NODE_FILE}.")

print("Data preparation complete.")

In [ ]:
# Cell 02: Load curated event-node dataset and define with-SEASON feature set
DATA_FILE = "storm_node_dataset_14nodes_200km_50kt.csv"

df = pd.read_csv(DATA_FILE)

target_col = "disrupted"
group_col = "SID"

numeric_features = [
    "SEASON",
    "month",
    "duration_hours",
    "max_wind_kts",
    "mean_wind_kts",
    "mean_storm_speed",
    "max_storm_speed",
    "min_distance_km",
    "coastal"
]

categorical_features = [
    "node_type"
]

feature_cols = numeric_features + categorical_features

X = df[feature_cols].copy()
y = df[target_col].astype(int).copy()
groups = df[group_col].copy()

print("Dataset shape:", df.shape)
print("Number of hurricane events:", df["SID"].nunique())
print("Positive samples:", y.sum())
print("Positive ratio:", y.mean())

print("\nFeature columns:")
print(feature_cols)

df.head()

In [ ]:
# Cell 03: Define preprocessing pipeline and initial model pool
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    sparse_threshold=0
)


models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=SKLEARN_N_JOBS
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=800,
        random_state=42
    )
}


def expected_calibration_error(y_true, y_prob, n_bins=10):
    """
    Expected Calibration Error, ECE.
    越低越好。
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lower = bins[i]
        upper = bins[i + 1]

        if i == n_bins - 1:
            mask = (y_prob >= lower) & (y_prob <= upper)
        else:
            mask = (y_prob >= lower) & (y_prob < upper)

        if mask.sum() > 0:
            bin_accuracy = y_true[mask].mean()
            bin_confidence = y_prob[mask].mean()
            ece += (mask.sum() / len(y_prob)) * abs(bin_accuracy - bin_confidence)

    return ece


def safe_log_loss(y_true, y_prob):
    y_prob = np.clip(y_prob, 1e-6, 1 - 1e-6)
    return log_loss(y_true, y_prob)


def get_group_cv(n_splits=5, random_state=42):
    """
    优先使用 StratifiedGroupKFold。
    如果 sklearn 版本不支持，则退回 GroupKFold。
    """
    if HAS_STRATIFIED_GROUP_KFOLD:
        return StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state
        )
    else:
        return GroupKFold(n_splits=n_splits)


print("Preprocessor, models, and metrics ready.")

In [ ]:
# Cell 04: Event-level 5-fold cross-validation with SEASON
N_SPLITS = 5
cv = get_group_cv(n_splits=N_SPLITS, random_state=42)

cv_results = []
oof_predictions = {}

for model_name, model in models.items():
    print("=" * 80)
    print("Cross-validating:", model_name)

    oof_prob = np.zeros(len(df))

    fold_rows = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        clf = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", clone(model))
        ])

        clf.fit(X_train, y_train)
        y_prob = clf.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        oof_prob[test_idx] = y_prob

        fold_row = {
            "model": model_name,
            "fold": fold_id,
            "test_size": len(test_idx),
            "test_positive_samples": int(y_test.sum()),
            "test_positive_ratio": float(y_test.mean()),
            "roc_auc": roc_auc_score(y_test, y_prob),
            "average_precision": average_precision_score(y_test, y_prob),
            "brier_score": brier_score_loss(y_test, y_prob),
            "log_loss": safe_log_loss(y_test, y_prob),
            "ece": expected_calibration_error(y_test, y_prob, n_bins=10),
            "accuracy_at_05": accuracy_score(y_test, y_pred),
            "precision_at_05": precision_score(y_test, y_pred, zero_division=0),
            "recall_at_05": recall_score(y_test, y_pred, zero_division=0),
            "f1_at_05": f1_score(y_test, y_pred, zero_division=0)
        }

        fold_rows.append(fold_row)

        print(
            f"Fold {fold_id}: "
            f"pos={int(y_test.sum())}, "
            f"AUC={fold_row['roc_auc']:.4f}, "
            f"AP={fold_row['average_precision']:.4f}, "
            f"Brier={fold_row['brier_score']:.4f}"
        )

    oof_predictions[model_name] = oof_prob
    cv_results.extend(fold_rows)

cv_results_df = pd.DataFrame(cv_results)

cv_summary = (
    cv_results_df
    .groupby("model")
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        average_precision_mean=("average_precision", "mean"),
        average_precision_std=("average_precision", "std"),
        brier_score_mean=("brier_score", "mean"),
        brier_score_std=("brier_score", "std"),
        log_loss_mean=("log_loss", "mean"),
        ece_mean=("ece", "mean"),
        f1_at_05_mean=("f1_at_05", "mean"),
        recall_at_05_mean=("recall_at_05", "mean"),
        precision_at_05_mean=("precision_at_05", "mean")
    )
    .reset_index()
    .sort_values(
        by=["average_precision_mean", "roc_auc_mean"],
        ascending=False
    )
)

cv_results_df.to_csv("cv_fold_results_no_leakage.csv", index=False, encoding="utf-8-sig")
cv_summary.to_csv("cv_model_summary_no_leakage.csv", index=False, encoding="utf-8-sig")

cv_summary

In [ ]:
# Cell 05: Select best model from initial with-SEASON CV
selected_model_name = cv_summary.iloc[0]["model"]

print("Selected model based on CV summary:", selected_model_name)

cv_summary

In [ ]:
# Cell 06: Cross-fitted calibration for initial with-SEASON model
def valid_group_holdout_indices(
    X_part,
    y_part,
    groups_part,
    test_size=0.25,
    random_state_start=0,
    min_pos_train=5,
    min_pos_valid=3,
    max_attempts=500
):
    """
    在训练集内部按 group 再划分 fit/calibration。
    确保 fit 和 calibration 都有正样本。
    """
    for seed in range(random_state_start, random_state_start + max_attempts):
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed
        )

        fit_idx, calib_idx = next(splitter.split(X_part, y_part, groups_part))

        if (
            y_part.iloc[fit_idx].sum() >= min_pos_train
            and y_part.iloc[calib_idx].sum() >= min_pos_valid
        ):
            return fit_idx, calib_idx, seed

    raise ValueError("Could not find a valid fit/calibration split.")


def logit_transform(prob, eps=1e-6):
    prob = np.clip(np.asarray(prob), eps, 1 - eps)
    return np.log(prob / (1 - prob)).reshape(-1, 1)


base_model = models[selected_model_name]

cv = get_group_cv(n_splits=N_SPLITS, random_state=42)

oof_uncalibrated_prob = np.zeros(len(df))
oof_calibrated_prob = np.zeros(len(df))

calibration_fold_rows = []

for fold_id, (trainval_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
    print("=" * 80)
    print("Calibration fold:", fold_id)

    X_trainval = X.iloc[trainval_idx].reset_index(drop=True)
    y_trainval = y.iloc[trainval_idx].reset_index(drop=True)
    groups_trainval = groups.iloc[trainval_idx].reset_index(drop=True)

    fit_rel_idx, calib_rel_idx, calib_seed = valid_group_holdout_indices(
        X_trainval,
        y_trainval,
        groups_trainval,
        test_size=0.25,
        random_state_start=fold_id * 100,
        min_pos_train=5,
        min_pos_valid=3
    )

    fit_idx = trainval_idx[fit_rel_idx]
    calib_idx = trainval_idx[calib_rel_idx]

    X_fit = X.iloc[fit_idx].copy()
    y_fit = y.iloc[fit_idx].copy()

    X_calib = X.iloc[calib_idx].copy()
    y_calib = y.iloc[calib_idx].copy()

    X_test = X.iloc[test_idx].copy()
    y_test = y.iloc[test_idx].copy()

    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", clone(base_model))
    ])

    clf.fit(X_fit, y_fit)

    calib_prob_uncal = clf.predict_proba(X_calib)[:, 1]
    test_prob_uncal = clf.predict_proba(X_test)[:, 1]

    # Platt scaling: 用 calibration set 拟合 sigmoid calibration
    calibrator = LogisticRegression(max_iter=1000)
    calibrator.fit(logit_transform(calib_prob_uncal), y_calib)

    test_prob_cal = calibrator.predict_proba(
        logit_transform(test_prob_uncal)
    )[:, 1]

    oof_uncalibrated_prob[test_idx] = test_prob_uncal
    oof_calibrated_prob[test_idx] = test_prob_cal

    fold_row = {
        "fold": fold_id,
        "calibration_seed": calib_seed,
        "fit_positive_samples": int(y_fit.sum()),
        "calib_positive_samples": int(y_calib.sum()),
        "test_positive_samples": int(y_test.sum()),

        "uncalibrated_auc": roc_auc_score(y_test, test_prob_uncal),
        "uncalibrated_ap": average_precision_score(y_test, test_prob_uncal),
        "uncalibrated_brier": brier_score_loss(y_test, test_prob_uncal),
        "uncalibrated_log_loss": safe_log_loss(y_test, test_prob_uncal),
        "uncalibrated_ece": expected_calibration_error(y_test, test_prob_uncal),

        "calibrated_auc": roc_auc_score(y_test, test_prob_cal),
        "calibrated_ap": average_precision_score(y_test, test_prob_cal),
        "calibrated_brier": brier_score_loss(y_test, test_prob_cal),
        "calibrated_log_loss": safe_log_loss(y_test, test_prob_cal),
        "calibrated_ece": expected_calibration_error(y_test, test_prob_cal)
    }

    calibration_fold_rows.append(fold_row)

    print(
        f"Uncalibrated: AP={fold_row['uncalibrated_ap']:.4f}, "
        f"Brier={fold_row['uncalibrated_brier']:.4f}, "
        f"ECE={fold_row['uncalibrated_ece']:.4f}"
    )
    print(
        f"Calibrated:   AP={fold_row['calibrated_ap']:.4f}, "
        f"Brier={fold_row['calibrated_brier']:.4f}, "
        f"ECE={fold_row['calibrated_ece']:.4f}"
    )

calibration_fold_df = pd.DataFrame(calibration_fold_rows)

calibration_summary = pd.DataFrame([
    {
        "model": selected_model_name,
        "version": "uncalibrated_oof",
        "roc_auc": roc_auc_score(y, oof_uncalibrated_prob),
        "average_precision": average_precision_score(y, oof_uncalibrated_prob),
        "brier_score": brier_score_loss(y, oof_uncalibrated_prob),
        "log_loss": safe_log_loss(y, oof_uncalibrated_prob),
        "ece": expected_calibration_error(y, oof_uncalibrated_prob)
    },
    {
        "model": selected_model_name,
        "version": "calibrated_oof",
        "roc_auc": roc_auc_score(y, oof_calibrated_prob),
        "average_precision": average_precision_score(y, oof_calibrated_prob),
        "brier_score": brier_score_loss(y, oof_calibrated_prob),
        "log_loss": safe_log_loss(y, oof_calibrated_prob),
        "ece": expected_calibration_error(y, oof_calibrated_prob)
    }
])

calibration_fold_df.to_csv(
    "calibration_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

calibration_summary.to_csv(
    "calibration_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

calibration_summary

In [ ]:
# Cell 07: Save initial calibrated OOF risk predictions
risk_predictions_oof = df[
    [
        "SID",
        "SEASON",
        "NAME",
        "node_id",
        "node_name",
        "node_type",
        "month",
        "max_wind_kts",
        "mean_wind_kts",
        "min_distance_km",
        "coastal",
        "disrupted"
    ]
].copy()

risk_predictions_oof["predicted_probability_uncalibrated_oof"] = oof_uncalibrated_prob
risk_predictions_oof["predicted_probability_calibrated_oof"] = oof_calibrated_prob

risk_predictions_oof.to_csv(
    "risk_predictions_oof_calibrated.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: risk_predictions_oof_calibrated.csv")
risk_predictions_oof.head()

In [ ]:
# Cell 08: Plot calibration curve for initial calibrated model
prob_true_uncal, prob_pred_uncal = calibration_curve(
    y,
    oof_uncalibrated_prob,
    n_bins=10,
    strategy="quantile"
)

prob_true_cal, prob_pred_cal = calibration_curve(
    y,
    oof_calibrated_prob,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.plot(prob_pred_uncal, prob_true_uncal, marker="o", label="Uncalibrated")
plt.plot(prob_pred_cal, prob_true_cal, marker="o", label="Calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed exposure frequency")
plt.title("Calibration Curve")
plt.legend()
plt.tight_layout()
plt.savefig("calibration_curve.png", dpi=300)
plt.show()

In [ ]:
# Cell 09: Build calibration bin tables
# Calibration bin table: more reliable than the curve alone for rare events

def calibration_bin_table(y_true, y_prob, n_bins=10, strategy="quantile"):
    temp = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "y_prob": np.asarray(y_prob)
    })

    if strategy == "quantile":
        temp["bin"] = pd.qcut(
            temp["y_prob"],
            q=n_bins,
            duplicates="drop"
        )
    elif strategy == "uniform":
        temp["bin"] = pd.cut(
            temp["y_prob"],
            bins=np.linspace(0, 1, n_bins + 1),
            include_lowest=True
        )
    else:
        raise ValueError("strategy must be either 'quantile' or 'uniform'.")

    out = (
        temp.groupby("bin", observed=False)
        .agg(
            sample_count=("y_true", "count"),
            positive_count=("y_true", "sum"),
            observed_frequency=("y_true", "mean"),
            mean_predicted_probability=("y_prob", "mean"),
            min_predicted_probability=("y_prob", "min"),
            max_predicted_probability=("y_prob", "max")
        )
        .reset_index()
    )

    out["absolute_calibration_error"] = (
        out["observed_frequency"] - out["mean_predicted_probability"]
    ).abs()

    return out


calibration_bins_quantile = calibration_bin_table(
    y,
    oof_calibrated_prob,
    n_bins=10,
    strategy="quantile"
)

calibration_bins_uniform = calibration_bin_table(
    y,
    oof_calibrated_prob,
    n_bins=10,
    strategy="uniform"
)

calibration_bins_quantile.to_csv(
    "calibration_bins_quantile.csv",
    index=False,
    encoding="utf-8-sig"
)

calibration_bins_uniform.to_csv(
    "calibration_bins_uniform.csv",
    index=False,
    encoding="utf-8-sig"
)

calibration_bins_quantile

In [ ]:
# Cell 10: Permutation importance on held-out group split
def get_valid_outer_split(X, y, groups, test_size=0.2, min_pos_test=10, max_attempts=500):
    for seed in range(max_attempts):
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed
        )

        train_idx, test_idx = next(splitter.split(X, y, groups))

        if y.iloc[test_idx].sum() >= min_pos_test and y.iloc[train_idx].sum() >= 20:
            return train_idx, test_idx, seed

    raise ValueError("Could not find valid outer split.")


train_idx, test_idx, seed = get_valid_outer_split(
    X,
    y,
    groups,
    test_size=0.2,
    min_pos_test=10
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

importance_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", clone(base_model))
])

importance_model.fit(X_train, y_train)

print("Held-out split seed:", seed)
print("Test positive samples:", int(y_test.sum()))
print("Test positive ratio:", y_test.mean())

# Key fix: n_jobs=1
perm_result = permutation_importance(
    importance_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=30,
    random_state=42,
    n_jobs=1
)

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

feature_importance.to_csv(
    "permutation_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

feature_importance

In [ ]:
# Cell 11: Plot permutation importance from held-out split
top_k = min(12, len(feature_importance))

plot_df = feature_importance.head(top_k).sort_values(
    by="importance_mean",
    ascending=True
)

plt.figure(figsize=(7, 5))
plt.barh(
    plot_df["feature"],
    plot_df["importance_mean"],
    xerr=plot_df["importance_std"]
)
plt.xlabel("Permutation importance based on Average Precision")
plt.ylabel("Feature")
plt.title("Feature Importance for Disruption Exposure Prediction")
plt.tight_layout()
plt.savefig("permutation_importance.png", dpi=300)
plt.show()

In [ ]:
# Cell 12: Build initial full scenario-node probability matrix
scenario_node_probabilities = risk_predictions_oof[
    [
        "SID",
        "SEASON",
        "NAME",
        "node_id",
        "node_name",
        "node_type",
        "predicted_probability_uncalibrated_oof",
        "predicted_probability_calibrated_oof",
        "disrupted"
    ]
].copy()

scenario_node_probabilities = scenario_node_probabilities.rename(columns={
    "SID": "scenario_id",
    "predicted_probability_calibrated_oof": "p_is_calibrated",
    "predicted_probability_uncalibrated_oof": "p_is_uncalibrated"
})

num_scenarios = scenario_node_probabilities["scenario_id"].nunique()

scenario_node_probabilities["scenario_weight"] = 1.0 / num_scenarios

scenario_node_probabilities.to_csv(
    "scenario_node_probabilities_full.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Number of scenarios:", num_scenarios)
print("Saved: scenario_node_probabilities_full.csv")

scenario_node_probabilities.head()

In [ ]:
# Cell 13: Generate KMeans reduced scenarios K=20
SCENARIO_K = 20

scenario_summary = (
    risk_predictions_oof
    .groupby(["SID", "SEASON", "NAME"])
    .agg(
        max_predicted_risk=("predicted_probability_calibrated_oof", "max"),
        mean_predicted_risk=("predicted_probability_calibrated_oof", "mean"),
        total_predicted_risk=("predicted_probability_calibrated_oof", "sum"),
        observed_exposure_count=("disrupted", "sum"),
        max_wind_kts=("max_wind_kts", "max"),
        mean_wind_kts=("mean_wind_kts", "mean"),
        min_distance_to_network=("min_distance_km", "min"),
        month=("month", "first")
    )
    .reset_index()
)

scenario_features = [
    "max_predicted_risk",
    "mean_predicted_risk",
    "total_predicted_risk",
    "observed_exposure_count",
    "max_wind_kts",
    "mean_wind_kts",
    "min_distance_to_network",
    "month"
]

scaler = StandardScaler()
scenario_X = scaler.fit_transform(scenario_summary[scenario_features])

kmeans = KMeans(
    n_clusters=SCENARIO_K,
    random_state=42,
    n_init=20
)

scenario_summary["cluster"] = kmeans.fit_predict(scenario_X)

representative_rows = []

for cluster_id in sorted(scenario_summary["cluster"].unique()):
    cluster_indices = scenario_summary.index[
        scenario_summary["cluster"] == cluster_id
    ].tolist()

    cluster_vectors = scenario_X[cluster_indices]
    center = kmeans.cluster_centers_[cluster_id]

    distances_to_center = np.linalg.norm(cluster_vectors - center, axis=1)
    medoid_local_pos = int(np.argmin(distances_to_center))
    medoid_index = cluster_indices[medoid_local_pos]

    row = scenario_summary.loc[medoid_index].copy()
    row["cluster_size"] = len(cluster_indices)
    row["scenario_weight"] = len(cluster_indices) / len(scenario_summary)

    representative_rows.append(row)

reduced_scenarios = pd.DataFrame(representative_rows)

reduced_scenarios = reduced_scenarios.sort_values(
    by="scenario_weight",
    ascending=False
).reset_index(drop=True)

reduced_scenarios.to_csv(
    "reduced_hurricane_scenarios_K20.csv",
    index=False,
    encoding="utf-8-sig"
)

selected_scenario_ids = set(reduced_scenarios["SID"])
scenario_weights = reduced_scenarios.set_index("SID")["scenario_weight"].to_dict()

scenario_node_probabilities_reduced = scenario_node_probabilities[
    scenario_node_probabilities["scenario_id"].isin(selected_scenario_ids)
].copy()

scenario_node_probabilities_reduced["scenario_weight"] = (
    scenario_node_probabilities_reduced["scenario_id"].map(scenario_weights)
)

scenario_node_probabilities_reduced.to_csv(
    "scenario_node_probabilities_reduced_K20.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Reduced scenario count:", reduced_scenarios.shape[0])
print("Weight sum:", reduced_scenarios["scenario_weight"].sum())

reduced_scenarios

In [ ]:
# Cell 14: Define without-SEASON feature set
numeric_features_no_season = [
    "month",
    "duration_hours",
    "max_wind_kts",
    "mean_wind_kts",
    "mean_storm_speed",
    "max_storm_speed",
    "min_distance_km",
    "coastal"
]

categorical_features_no_season = [
    "node_type"
]

feature_cols_no_season = numeric_features_no_season + categorical_features_no_season

X_no_season = df[feature_cols_no_season].copy()
y_no_season = df[target_col].astype(int).copy()
groups_no_season = df[group_col].copy()

try:
    encoder_no_season = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder_no_season = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer_no_season = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_no_season = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder_no_season)
])

preprocessor_no_season = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_no_season, numeric_features_no_season),
        ("cat", categorical_transformer_no_season, categorical_features_no_season)
    ],
    sparse_threshold=0
)

print("No-SEASON feature set:")
print(feature_cols_no_season)
print("X_no_season shape:", X_no_season.shape)

In [ ]:
# Cell 15: Event-level 5-fold cross-validation without SEASON
cv_no_season = get_group_cv(n_splits=N_SPLITS, random_state=42)

cv_results_no_season = []

for model_name, model in models.items():
    print("=" * 80)
    print("Cross-validating without SEASON:", model_name)

    for fold_id, (train_idx, test_idx) in enumerate(
        cv_no_season.split(X_no_season, y_no_season, groups_no_season),
        start=1
    ):
        X_train = X_no_season.iloc[train_idx].copy()
        X_test = X_no_season.iloc[test_idx].copy()
        y_train = y_no_season.iloc[train_idx].copy()
        y_test = y_no_season.iloc[test_idx].copy()

        clf = Pipeline(steps=[
            ("preprocessor", preprocessor_no_season),
            ("model", clone(model))
        ])

        clf.fit(X_train, y_train)

        y_prob = clf.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        row = {
            "model": model_name,
            "fold": fold_id,
            "test_size": len(test_idx),
            "test_positive_samples": int(y_test.sum()),
            "test_positive_ratio": float(y_test.mean()),
            "roc_auc": roc_auc_score(y_test, y_prob),
            "average_precision": average_precision_score(y_test, y_prob),
            "brier_score": brier_score_loss(y_test, y_prob),
            "log_loss": safe_log_loss(y_test, y_prob),
            "ece": expected_calibration_error(y_test, y_prob, n_bins=10),
            "accuracy_at_05": accuracy_score(y_test, y_pred),
            "precision_at_05": precision_score(y_test, y_pred, zero_division=0),
            "recall_at_05": recall_score(y_test, y_pred, zero_division=0),
            "f1_at_05": f1_score(y_test, y_pred, zero_division=0)
        }

        cv_results_no_season.append(row)

        print(
            f"Fold {fold_id}: "
            f"pos={int(y_test.sum())}, "
            f"AUC={row['roc_auc']:.4f}, "
            f"AP={row['average_precision']:.4f}, "
            f"Brier={row['brier_score']:.4f}"
        )

cv_results_no_season_df = pd.DataFrame(cv_results_no_season)

cv_summary_no_season = (
    cv_results_no_season_df
    .groupby("model")
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        average_precision_mean=("average_precision", "mean"),
        average_precision_std=("average_precision", "std"),
        brier_score_mean=("brier_score", "mean"),
        brier_score_std=("brier_score", "std"),
        log_loss_mean=("log_loss", "mean"),
        ece_mean=("ece", "mean"),
        f1_at_05_mean=("f1_at_05", "mean"),
        recall_at_05_mean=("recall_at_05", "mean"),
        precision_at_05_mean=("precision_at_05", "mean")
    )
    .reset_index()
    .sort_values(
        by=["average_precision_mean", "roc_auc_mean"],
        ascending=False
    )
)

cv_results_no_season_df.to_csv(
    "cv_fold_results_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

cv_summary_no_season.to_csv(
    "cv_model_summary_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

cv_summary_no_season

In [ ]:
# Cell 16: Compare with-SEASON and without-SEASON performance
cv_summary_with_season = cv_summary.copy()
cv_summary_with_season["feature_set"] = "with_SEASON"

cv_summary_no_season_compare = cv_summary_no_season.copy()
cv_summary_no_season_compare["feature_set"] = "without_SEASON"

compare_season_effect = pd.concat(
    [cv_summary_with_season, cv_summary_no_season_compare],
    ignore_index=True
)

compare_season_effect = compare_season_effect[
    [
        "feature_set",
        "model",
        "roc_auc_mean",
        "roc_auc_std",
        "average_precision_mean",
        "average_precision_std",
        "brier_score_mean",
        "log_loss_mean",
        "ece_mean",
        "f1_at_05_mean",
        "recall_at_05_mean",
        "precision_at_05_mean"
    ]
].sort_values(
    by=["model", "feature_set"]
)

compare_season_effect.to_csv(
    "compare_with_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

compare_season_effect

In [ ]:
# Cell 17: Define final without-SEASON feature set and Random Forest
# Final decision:
# We exclude SEASON to avoid temporal shortcut learning.
# The final predictive model uses physically interpretable hazard-exposure features.

final_numeric_features = [
    "month",
    "duration_hours",
    "max_wind_kts",
    "mean_wind_kts",
    "mean_storm_speed",
    "max_storm_speed",
    "min_distance_km",
    "coastal"
]

final_categorical_features = [
    "node_type"
]

final_feature_cols = final_numeric_features + final_categorical_features

X_final = df[final_feature_cols].copy()
y_final = df[target_col].astype(int).copy()
groups_final = df[group_col].copy()

try:
    final_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    final_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

final_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

final_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", final_encoder)
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ("num", final_numeric_transformer, final_numeric_features),
        ("cat", final_categorical_transformer, final_categorical_features)
    ],
    sparse_threshold=0
)

final_base_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=SKLEARN_N_JOBS
)

print("Final feature set:")
print(final_feature_cols)
print("\nX_final shape:", X_final.shape)
print("Positive samples:", int(y_final.sum()))
print("Positive ratio:", y_final.mean())

In [ ]:
# Cell 18: Cross-fitted probability calibration for final without-SEASON model
def final_expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lower = bins[i]
        upper = bins[i + 1]

        if i == n_bins - 1:
            mask = (y_prob >= lower) & (y_prob <= upper)
        else:
            mask = (y_prob >= lower) & (y_prob < upper)

        if mask.sum() > 0:
            bin_accuracy = y_true[mask].mean()
            bin_confidence = y_prob[mask].mean()
            ece += (mask.sum() / len(y_prob)) * abs(bin_accuracy - bin_confidence)

    return ece


def final_safe_log_loss(y_true, y_prob):
    y_prob = np.clip(y_prob, 1e-6, 1 - 1e-6)
    return log_loss(y_true, y_prob)


def final_valid_group_holdout_indices(
    X_part,
    y_part,
    groups_part,
    test_size=0.25,
    random_state_start=0,
    min_pos_train=5,
    min_pos_valid=3,
    max_attempts=500
):
    for seed in range(random_state_start, random_state_start + max_attempts):
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed
        )

        fit_idx, calib_idx = next(splitter.split(X_part, y_part, groups_part))

        if (
            y_part.iloc[fit_idx].sum() >= min_pos_train
            and y_part.iloc[calib_idx].sum() >= min_pos_valid
        ):
            return fit_idx, calib_idx, seed

    raise ValueError("Could not find a valid fit/calibration split.")


def final_logit_transform(prob, eps=1e-6):
    prob = np.clip(np.asarray(prob), eps, 1 - eps)
    return np.log(prob / (1 - prob)).reshape(-1, 1)


N_SPLITS_FINAL = 5

try:
    final_cv = StratifiedGroupKFold(
        n_splits=N_SPLITS_FINAL,
        shuffle=True,
        random_state=42
    )
except NameError:
    final_cv = GroupKFold(n_splits=N_SPLITS_FINAL)

oof_prob_uncalibrated_final = np.zeros(len(df))
oof_prob_calibrated_final = np.zeros(len(df))

final_calibration_rows = []

for fold_id, (trainval_idx, test_idx) in enumerate(
    final_cv.split(X_final, y_final, groups_final),
    start=1
):
    print("=" * 80)
    print("Final calibration fold:", fold_id)

    X_trainval = X_final.iloc[trainval_idx].reset_index(drop=True)
    y_trainval = y_final.iloc[trainval_idx].reset_index(drop=True)
    groups_trainval = groups_final.iloc[trainval_idx].reset_index(drop=True)

    fit_rel_idx, calib_rel_idx, calib_seed = final_valid_group_holdout_indices(
        X_trainval,
        y_trainval,
        groups_trainval,
        test_size=0.25,
        random_state_start=fold_id * 100,
        min_pos_train=5,
        min_pos_valid=3
    )

    fit_idx = trainval_idx[fit_rel_idx]
    calib_idx = trainval_idx[calib_rel_idx]

    X_fit = X_final.iloc[fit_idx].copy()
    y_fit = y_final.iloc[fit_idx].copy()

    X_calib = X_final.iloc[calib_idx].copy()
    y_calib = y_final.iloc[calib_idx].copy()

    X_test = X_final.iloc[test_idx].copy()
    y_test = y_final.iloc[test_idx].copy()

    clf = Pipeline(steps=[
        ("preprocessor", final_preprocessor),
        ("model", clone(final_base_model))
    ])

    clf.fit(X_fit, y_fit)

    calib_prob_uncal = clf.predict_proba(X_calib)[:, 1]
    test_prob_uncal = clf.predict_proba(X_test)[:, 1]

    calibrator = LogisticRegression(max_iter=1000)
    calibrator.fit(final_logit_transform(calib_prob_uncal), y_calib)

    test_prob_cal = calibrator.predict_proba(
        final_logit_transform(test_prob_uncal)
    )[:, 1]

    oof_prob_uncalibrated_final[test_idx] = test_prob_uncal
    oof_prob_calibrated_final[test_idx] = test_prob_cal

    row = {
        "fold": fold_id,
        "calibration_seed": calib_seed,
        "fit_positive_samples": int(y_fit.sum()),
        "calib_positive_samples": int(y_calib.sum()),
        "test_positive_samples": int(y_test.sum()),

        "uncalibrated_auc": roc_auc_score(y_test, test_prob_uncal),
        "uncalibrated_ap": average_precision_score(y_test, test_prob_uncal),
        "uncalibrated_brier": brier_score_loss(y_test, test_prob_uncal),
        "uncalibrated_log_loss": final_safe_log_loss(y_test, test_prob_uncal),
        "uncalibrated_ece": final_expected_calibration_error(y_test, test_prob_uncal),

        "calibrated_auc": roc_auc_score(y_test, test_prob_cal),
        "calibrated_ap": average_precision_score(y_test, test_prob_cal),
        "calibrated_brier": brier_score_loss(y_test, test_prob_cal),
        "calibrated_log_loss": final_safe_log_loss(y_test, test_prob_cal),
        "calibrated_ece": final_expected_calibration_error(y_test, test_prob_cal)
    }

    final_calibration_rows.append(row)

    print(
        f"Uncalibrated: AP={row['uncalibrated_ap']:.4f}, "
        f"Brier={row['uncalibrated_brier']:.4f}, "
        f"ECE={row['uncalibrated_ece']:.4f}"
    )

    print(
        f"Calibrated:   AP={row['calibrated_ap']:.4f}, "
        f"Brier={row['calibrated_brier']:.4f}, "
        f"ECE={row['calibrated_ece']:.4f}"
    )


final_calibration_fold_df = pd.DataFrame(final_calibration_rows)

final_calibration_summary = pd.DataFrame([
    {
        "model": "Random Forest",
        "feature_set": "without_SEASON",
        "version": "uncalibrated_oof",
        "roc_auc": roc_auc_score(y_final, oof_prob_uncalibrated_final),
        "average_precision": average_precision_score(y_final, oof_prob_uncalibrated_final),
        "brier_score": brier_score_loss(y_final, oof_prob_uncalibrated_final),
        "log_loss": final_safe_log_loss(y_final, oof_prob_uncalibrated_final),
        "ece": final_expected_calibration_error(y_final, oof_prob_uncalibrated_final)
    },
    {
        "model": "Random Forest",
        "feature_set": "without_SEASON",
        "version": "calibrated_oof",
        "roc_auc": roc_auc_score(y_final, oof_prob_calibrated_final),
        "average_precision": average_precision_score(y_final, oof_prob_calibrated_final),
        "brier_score": brier_score_loss(y_final, oof_prob_calibrated_final),
        "log_loss": final_safe_log_loss(y_final, oof_prob_calibrated_final),
        "ece": final_expected_calibration_error(y_final, oof_prob_calibrated_final)
    }
])

final_calibration_fold_df.to_csv(
    "final_calibration_fold_results_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

final_calibration_summary.to_csv(
    "final_calibration_summary_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

final_calibration_summary

In [ ]:
# Cell 19: Save final scenario-node OOF predictions
final_risk_predictions_oof = df[
    [
        "SID",
        "SEASON",
        "NAME",
        "node_id",
        "node_name",
        "node_type",
        "month",
        "max_wind_kts",
        "mean_wind_kts",
        "min_distance_km",
        "coastal",
        "disrupted"
    ]
].copy()

final_risk_predictions_oof["p_uncalibrated"] = oof_prob_uncalibrated_final
final_risk_predictions_oof["p_calibrated"] = oof_prob_calibrated_final

final_risk_predictions_oof.to_csv(
    "final_risk_predictions_oof_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: final_risk_predictions_oof_without_season.csv")
final_risk_predictions_oof.head()

In [ ]:
# Cell 20: Build final full scenario-node probability matrix
final_scenario_node_probabilities_full = final_risk_predictions_oof[
    [
        "SID",
        "SEASON",
        "NAME",
        "node_id",
        "node_name",
        "node_type",
        "p_uncalibrated",
        "p_calibrated",
        "disrupted"
    ]
].copy()

final_scenario_node_probabilities_full = final_scenario_node_probabilities_full.rename(columns={
    "SID": "scenario_id",
    "p_calibrated": "p_is_calibrated",
    "p_uncalibrated": "p_is_uncalibrated"
})

num_final_scenarios = final_scenario_node_probabilities_full["scenario_id"].nunique()
final_scenario_node_probabilities_full["scenario_weight"] = 1.0 / num_final_scenarios

final_scenario_node_probabilities_full.to_csv(
    "final_scenario_node_probabilities_full_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Number of scenarios:", num_final_scenarios)
print("Saved: final_scenario_node_probabilities_full_without_season.csv")

final_scenario_node_probabilities_full.head()

In [ ]:
# Cell 21: Hybrid scenario reduction using final calibrated probabilities
SCENARIO_K_FINAL = 20
TOP_RISK_N_FINAL = 10

scenario_full_summary_final = (
    final_scenario_node_probabilities_full
    .groupby(["scenario_id", "SEASON", "NAME"])
    .agg(
        max_predicted_risk=("p_is_calibrated", "max"),
        mean_predicted_risk=("p_is_calibrated", "mean"),
        total_predicted_risk=("p_is_calibrated", "sum"),
        observed_exposure_count=("disrupted", "sum"),
        scenario_weight_original=("scenario_weight", "first")
    )
    .reset_index()
)

# Add physical storm-level variables for scenario reduction
storm_physical_summary = (
    final_risk_predictions_oof
    .groupby(["SID", "SEASON", "NAME"])
    .agg(
        max_wind_kts=("max_wind_kts", "max"),
        mean_wind_kts=("mean_wind_kts", "mean"),
        min_distance_to_network=("min_distance_km", "min"),
        month=("month", "first")
    )
    .reset_index()
    .rename(columns={"SID": "scenario_id"})
)

scenario_full_summary_final = scenario_full_summary_final.merge(
    storm_physical_summary,
    on=["scenario_id", "SEASON", "NAME"],
    how="left"
)

scenario_features_final = [
    "max_predicted_risk",
    "mean_predicted_risk",
    "total_predicted_risk",
    "observed_exposure_count",
    "max_wind_kts",
    "mean_wind_kts",
    "min_distance_to_network",
    "month"
]

scenario_scaler_final = StandardScaler()
scenario_X_final = scenario_scaler_final.fit_transform(
    scenario_full_summary_final[scenario_features_final]
)

kmeans_final = KMeans(
    n_clusters=SCENARIO_K_FINAL,
    random_state=42,
    n_init=20
)

scenario_full_summary_final["cluster"] = kmeans_final.fit_predict(scenario_X_final)

representative_rows_final = []

for cluster_id in sorted(scenario_full_summary_final["cluster"].unique()):
    cluster_indices = scenario_full_summary_final.index[
        scenario_full_summary_final["cluster"] == cluster_id
    ].tolist()

    cluster_vectors = scenario_X_final[cluster_indices]
    center = kmeans_final.cluster_centers_[cluster_id]

    distances_to_center = np.linalg.norm(cluster_vectors - center, axis=1)
    medoid_local_pos = int(np.argmin(distances_to_center))
    medoid_index = cluster_indices[medoid_local_pos]

    row = scenario_full_summary_final.loc[medoid_index].copy()
    row["cluster_size"] = len(cluster_indices)
    row["scenario_weight"] = len(cluster_indices) / len(scenario_full_summary_final)
    row["selection_source"] = "kmeans_representative"

    representative_rows_final.append(row)

kmeans_reduced_final = pd.DataFrame(representative_rows_final)

top_risk_scenarios_final = (
    scenario_full_summary_final
    .sort_values(
        by=["total_predicted_risk", "max_predicted_risk"],
        ascending=False
    )
    .head(TOP_RISK_N_FINAL)
    .copy()
)

top_risk_scenarios_final["scenario_weight"] = (
    top_risk_scenarios_final["scenario_weight_original"]
)

top_risk_scenarios_final["selection_source"] = "tail_risk"

hybrid_scenarios_final = pd.concat(
    [
        kmeans_reduced_final,
        top_risk_scenarios_final
    ],
    ignore_index=True
)

hybrid_scenarios_final = (
    hybrid_scenarios_final
    .groupby(["scenario_id", "SEASON", "NAME"], as_index=False)
    .agg(
        max_predicted_risk=("max_predicted_risk", "max"),
        mean_predicted_risk=("mean_predicted_risk", "max"),
        total_predicted_risk=("total_predicted_risk", "max"),
        observed_exposure_count=("observed_exposure_count", "max"),
        max_wind_kts=("max_wind_kts", "max"),
        mean_wind_kts=("mean_wind_kts", "max"),
        min_distance_to_network=("min_distance_to_network", "min"),
        month=("month", "first"),
        scenario_weight=("scenario_weight", "sum"),
        selection_source=("selection_source", lambda x: "+".join(sorted(set(x))))
    )
)

hybrid_scenarios_final["scenario_weight"] = (
    hybrid_scenarios_final["scenario_weight"]
    / hybrid_scenarios_final["scenario_weight"].sum()
)

hybrid_scenarios_final = hybrid_scenarios_final.sort_values(
    by=["selection_source", "total_predicted_risk"],
    ascending=[True, False]
).reset_index(drop=True)

hybrid_scenarios_final.to_csv(
    "final_hybrid_reduced_hurricane_scenarios_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

selected_final_ids = set(hybrid_scenarios_final["scenario_id"])
final_hybrid_weights = hybrid_scenarios_final.set_index("scenario_id")["scenario_weight"].to_dict()

final_scenario_node_probabilities_hybrid = final_scenario_node_probabilities_full[
    final_scenario_node_probabilities_full["scenario_id"].isin(selected_final_ids)
].copy()

final_scenario_node_probabilities_hybrid["scenario_weight"] = (
    final_scenario_node_probabilities_hybrid["scenario_id"].map(final_hybrid_weights)
)

final_scenario_node_probabilities_hybrid.to_csv(
    "final_scenario_node_probabilities_hybrid_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

print("KMeans scenario count:", kmeans_reduced_final.shape[0])
print("Top-risk scenario count:", top_risk_scenarios_final.shape[0])
print("Hybrid unique scenario count:", hybrid_scenarios_final.shape[0])
print("Hybrid weight sum:", hybrid_scenarios_final["scenario_weight"].sum())

hybrid_scenarios_final

In [ ]:
# Cell 22: Build final optimization scenario-node risk matrix
final_optimization_risk_matrix = final_scenario_node_probabilities_hybrid[
    [
        "scenario_id",
        "SEASON",
        "NAME",
        "node_id",
        "node_name",
        "node_type",
        "p_is_calibrated",
        "p_is_uncalibrated",
        "disrupted",
        "scenario_weight"
    ]
].copy()

final_optimization_risk_matrix = final_optimization_risk_matrix.rename(columns={
    "p_is_calibrated": "p_is",
    "p_is_uncalibrated": "p_is_sensitivity"
})

final_optimization_risk_matrix.to_csv(
    "final_optimization_scenario_node_risk_matrix_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: final_optimization_scenario_node_risk_matrix_without_season.csv")
print("Optimization risk matrix shape:", final_optimization_risk_matrix.shape)
print("Scenario count:", final_optimization_risk_matrix["scenario_id"].nunique())
print("Node count:", final_optimization_risk_matrix["node_id"].nunique())
print(
    "Scenario weight sum:",
    final_optimization_risk_matrix.drop_duplicates("scenario_id")["scenario_weight"].sum()
)

final_optimization_risk_matrix.head()

In [ ]:
# Cell 23: Import additional ML baseline models
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# Robust optional imports. This avoids NameError even if Cell 00 was modified or run in isolation.
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception as e:
    HAS_XGBOOST = False
    XGBClassifier = None
    print("XGBoost is not available; XGBoost benchmark will be skipped.")
    print("Reason:", repr(e))

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except Exception as e:
    HAS_LIGHTGBM = False
    LGBMClassifier = None
    print("LightGBM is not available; LightGBM benchmark will be skipped.")
    print("Reason:", repr(e))

print("Additional ML model imports checked.")
print("HAS_XGBOOST:", HAS_XGBOOST)
print("HAS_LIGHTGBM:", HAS_LIGHTGBM)


In [ ]:
# Cell 24: Load final dataset for seven-model benchmark
import pandas as pd
import numpy as np

DATA_FILE = "storm_node_dataset_14nodes_200km_50kt.csv"

df_model_compare = pd.read_csv(DATA_FILE)

target_col = "disrupted"
group_col = "SID"

# Final leakage-controlled feature set without SEASON
numeric_features_compare = [
    "month",
    "duration_hours",
    "max_wind_kts",
    "mean_wind_kts",
    "mean_storm_speed",
    "max_storm_speed",
    "min_distance_km",
    "coastal"
]

categorical_features_compare = [
    "node_type"
]

feature_cols_compare = numeric_features_compare + categorical_features_compare

X_compare = df_model_compare[feature_cols_compare].copy()
y_compare = df_model_compare[target_col].astype(int).copy()
groups_compare = df_model_compare[group_col].copy()

print("Dataset shape:", df_model_compare.shape)
print("Positive samples:", int(y_compare.sum()))
print("Positive ratio:", y_compare.mean())
print("Number of hurricane events:", df_model_compare[group_col].nunique())
print("Feature set:", feature_cols_compare)

In [ ]:
# Cell 25: Define preprocessing and metrics for seven-model benchmark
from sklearn.model_selection import GroupKFold

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except ImportError:
    HAS_STRATIFIED_GROUP_KFOLD = False

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

try:
    encoder_compare = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder_compare = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer_compare = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_compare = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder_compare)
])

preprocessor_compare = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_compare, numeric_features_compare),
        ("cat", categorical_transformer_compare, categorical_features_compare)
    ],
    sparse_threshold=0
)

def get_final_group_cv(n_splits=5, random_state=42):
    if HAS_STRATIFIED_GROUP_KFOLD:
        return StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state
        )
    else:
        return GroupKFold(n_splits=n_splits)

def expected_calibration_error_compare(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lower = bins[i]
        upper = bins[i + 1]

        if i == n_bins - 1:
            mask = (y_prob >= lower) & (y_prob <= upper)
        else:
            mask = (y_prob >= lower) & (y_prob < upper)

        if mask.sum() > 0:
            bin_accuracy = y_true[mask].mean()
            bin_confidence = y_prob[mask].mean()
            ece += (mask.sum() / len(y_prob)) * abs(bin_accuracy - bin_confidence)

    return ece

def safe_log_loss_compare(y_true, y_prob):
    y_prob = np.clip(y_prob, 1e-6, 1 - 1e-6)
    return log_loss(y_true, y_prob)

print("Preprocessor and metrics are ready.")
print("StratifiedGroupKFold available:", HAS_STRATIFIED_GROUP_KFOLD)

In [ ]:
# Cell 26: Define seven-model baseline pool
def make_model_pool(y_train):
    """
    Create models for each fold.
    XGBoost uses fold-specific scale_pos_weight.
    """
    positive_count = int(y_train.sum())
    negative_count = int(len(y_train) - positive_count)
    scale_pos_weight = negative_count / max(positive_count, 1)

    model_pool = {
        "Logistic Regression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ),

        "SVM": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=SKLEARN_N_JOBS
        ),

        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            random_state=42
        ),

        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=800,
            random_state=42
        )
    }

    if HAS_XGBOOST:
        model_pool["XGBoost"] = XGBClassifier(
            n_estimators=400,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=SKLEARN_N_JOBS
        )

    if HAS_LIGHTGBM:
        model_pool["LightGBM"] = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.03,
            max_depth=-1,
            num_leaves=31,
            class_weight="balanced",
            random_state=42,
            n_jobs=SKLEARN_N_JOBS,
            verbose=-1
        )

    return model_pool

print("Final ML model pool defined.")

In [ ]:
# Cell 27: Run event-level 5-fold benchmark for seven models
N_SPLITS_COMPARE = 5
cv_compare = get_final_group_cv(n_splits=N_SPLITS_COMPARE, random_state=42)

model_comparison_rows = []

# Store out-of-fold probabilities for optional downstream robustness checks
model_oof_probabilities = {}

model_names_reference = [
    "Logistic Regression",
    "SVM",
    "Random Forest",
    "Gradient Boosting",
    "MLP"
]
if HAS_XGBOOST:
    model_names_reference.append("XGBoost")
if HAS_LIGHTGBM:
    model_names_reference.append("LightGBM")

for model_name in model_names_reference:
    model_oof_probabilities[model_name] = np.zeros(len(df_model_compare))

for fold_id, (train_idx, test_idx) in enumerate(
    cv_compare.split(X_compare, y_compare, groups_compare),
    start=1
):
    print("=" * 100)
    print(f"Fold {fold_id}/{N_SPLITS_COMPARE}")

    X_train = X_compare.iloc[train_idx].copy()
    X_test = X_compare.iloc[test_idx].copy()

    y_train = y_compare.iloc[train_idx].copy()
    y_test = y_compare.iloc[test_idx].copy()

    model_pool = make_model_pool(y_train)

    print("Train positives:", int(y_train.sum()), "Test positives:", int(y_test.sum()))

    for model_name, model in model_pool.items():
        print(f"Training {model_name}...")

        clf = Pipeline(steps=[
            ("preprocessor", preprocessor_compare),
            ("model", model)
        ])

        clf.fit(X_train, y_train)

        y_prob = clf.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        model_oof_probabilities[model_name][test_idx] = y_prob

        row = {
            "model": model_name,
            "fold": fold_id,
            "test_size": len(test_idx),
            "test_positive_samples": int(y_test.sum()),
            "test_positive_ratio": float(y_test.mean()),

            "roc_auc": roc_auc_score(y_test, y_prob),
            "average_precision": average_precision_score(y_test, y_prob),
            "brier_score": brier_score_loss(y_test, y_prob),
            "log_loss": safe_log_loss_compare(y_test, y_prob),
            "ece": expected_calibration_error_compare(y_test, y_prob, n_bins=10),

            "accuracy_at_05": accuracy_score(y_test, y_pred),
            "precision_at_05": precision_score(y_test, y_pred, zero_division=0),
            "recall_at_05": recall_score(y_test, y_pred, zero_division=0),
            "f1_at_05": f1_score(y_test, y_pred, zero_division=0)
        }

        model_comparison_rows.append(row)

model_comparison_fold_df = pd.DataFrame(model_comparison_rows)

model_comparison_fold_df.to_csv(
    "final_model_comparison_fold_results_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

model_comparison_fold_df.head()

In [ ]:
# Cell 28: Summarize final model comparison
model_comparison_summary = (
    model_comparison_fold_df
    .groupby("model")
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        average_precision_mean=("average_precision", "mean"),
        average_precision_std=("average_precision", "std"),
        brier_score_mean=("brier_score", "mean"),
        brier_score_std=("brier_score", "std"),
        log_loss_mean=("log_loss", "mean"),
        ece_mean=("ece", "mean"),
        f1_at_05_mean=("f1_at_05", "mean"),
        recall_at_05_mean=("recall_at_05", "mean"),
        precision_at_05_mean=("precision_at_05", "mean")
    )
    .reset_index()
)

# Ranking logic:
# For rare-event prediction, AP is more informative than accuracy.
# Brier and ECE reflect probability quality.
model_comparison_summary = model_comparison_summary.sort_values(
    by=[
        "average_precision_mean",
        "roc_auc_mean",
        "brier_score_mean",
        "ece_mean"
    ],
    ascending=[False, False, True, True]
).reset_index(drop=True)

model_comparison_summary.to_csv(
    "final_model_comparison_summary_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

model_comparison_summary

In [ ]:
# Cell 29: Generate model selection diagnostic table
best_model_by_ap = model_comparison_summary.iloc[0]["model"]

rf_row = model_comparison_summary[
    model_comparison_summary["model"] == "Random Forest"
].iloc[0]

best_row = model_comparison_summary.iloc[0]

model_selection_diagnostic = pd.DataFrame([
    {
        "criterion": "Best model by Average Precision",
        "selected_model": best_model_by_ap,
        "average_precision_mean": best_row["average_precision_mean"],
        "roc_auc_mean": best_row["roc_auc_mean"],
        "brier_score_mean": best_row["brier_score_mean"],
        "ece_mean": best_row["ece_mean"]
    },
    {
        "criterion": "Random Forest reference",
        "selected_model": "Random Forest",
        "average_precision_mean": rf_row["average_precision_mean"],
        "roc_auc_mean": rf_row["roc_auc_mean"],
        "brier_score_mean": rf_row["brier_score_mean"],
        "ece_mean": rf_row["ece_mean"]
    }
])

model_selection_diagnostic.to_csv(
    "final_model_selection_diagnostic_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

model_selection_diagnostic

In [ ]:
# Cell 30: Permutation importance for final without-SEASON Random Forest
# Cell 30: Permutation importance for final without-SEASON Random Forest

final_pipe_without_season = Pipeline(steps=[
    ("preprocessor", final_preprocessor),
    ("model", clone(final_base_model))
])

final_pipe_without_season.fit(X_final, y_final)

perm_without_season = permutation_importance(
    final_pipe_without_season,
    X_final,
    y_final,
    scoring="average_precision",
    n_repeats=20,
    random_state=42,
    n_jobs=SKLEARN_N_JOBS
)

permutation_importance_without_season = pd.DataFrame({
    "feature": final_feature_cols,
    "importance_mean": perm_without_season.importances_mean,
    "importance_std": perm_without_season.importances_std
}).sort_values(by="importance_mean", ascending=False)

permutation_importance_without_season.to_csv(
    "permutation_importance_without_season.csv",
    index=False,
    encoding="utf-8-sig"
)

plot_df = permutation_importance_without_season.sort_values("importance_mean", ascending=True)

plt.figure(figsize=(7, 5))
plt.barh(
    plot_df["feature"],
    plot_df["importance_mean"],
    xerr=plot_df["importance_std"]
)
plt.xlabel("Permutation importance based on Average Precision")
plt.ylabel("Feature")
plt.title("Feature Importance for Final Without-SEASON Random Forest")
plt.tight_layout()
plt.savefig("permutation_importance_without_season.png", dpi=300)
plt.show()

permutation_importance_without_season

In [ ]:
# Cell 31: Output manifest for 01 data and ML pipeline
# Cell 31: Output manifest for 01 data and ML pipeline

manifest_files = [
    "ibtracs_NA_clean_2000.csv",
    "supply_chain_nodes_14.csv",
    "storm_node_dataset_14nodes_200km_50kt.csv",
    "threshold_summary_14nodes.csv",
    "compare_with_without_season.csv",
    "final_calibration_summary_without_season.csv",
    "final_risk_predictions_oof_without_season.csv",
    "final_scenario_node_probabilities_full_without_season.csv",
    "final_hybrid_reduced_hurricane_scenarios_without_season.csv",
    "final_scenario_node_probabilities_hybrid_without_season.csv",
    "final_optimization_scenario_node_risk_matrix_without_season.csv",
    "final_model_comparison_summary_without_season.csv",
    "final_model_selection_diagnostic_without_season.csv",
    "permutation_importance_without_season.csv",
    "permutation_importance_without_season.png",
]

manifest = []
for file in manifest_files:
    path = BASE_DIR / file
    manifest.append({
        "file": file,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan
    })

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv("output_manifest_01_ml_pipeline.csv", index=False, encoding="utf-8-sig")
manifest_df